# 🌸 Ejercicio: Clasificación de Iris con MLflow

## 🎯 Objetivos del Ejercicio

En este ejercicio práctico aprenderás a:

1. **Trabajar con un problema de clasificación** multiclase
2. **Aplicar MLflow** a un modelo de árbol de decisión
3. **Registrar experimentos** de forma profesional
4. **Evaluar y comparar** diferentes configuraciones
5. **Crear visualizaciones** para clasificación

---

## 🌺 El Dataset Iris

El **Iris Dataset** es uno de los datasets más famosos en Machine Learning, introducido por Ronald Fisher en 1936.

### 📊 Características del Dataset

| Aspecto | Descripción |
|---------|-------------|
| **Muestras** | 150 flores (50 por clase) |
| **Características** | 4 medidas físicas (cm) |
| **Clases** | 3 especies de iris |
| **Tipo** | Clasificación multiclase |
| **Dificultad** | ⭐⭐ (Principiante) |

### 🌸 Las 3 Especies de Iris

1. **Iris Setosa** (Clase 0)
2. **Iris Versicolor** (Clase 1)
3. **Iris Virginica** (Clase 2)

### 📏 Las 4 Características

1. **Sepal Length** (Longitud del sépalo)
2. **Sepal Width** (Ancho del sépalo)
3. **Petal Length** (Longitud del pétalo)
4. **Petal Width** (Ancho del pétalo)

---

## 🎯 Tu Misión

Construir un **Árbol de Decisión** que pueda clasificar correctamente las especies de iris basándose en sus medidas físicas, y documentar todo el proceso con MLflow.

---

## 🚀 ¡Empecemos!

In [0]:
import mlflow
import warnings
warnings.filterwarnings('ignore')


mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks")

# ⚠️ IMPORTANTE: Cambia esto por tu email de Databricks
email = ''  # Ejemplo: 'nombre.apellido@ejemplo.com'

# Validar que el email no esté vacío
if not email:
    print("⚠️  ADVERTENCIA: Debes configurar tu email antes de continuar")
    print("   Cambia la variable 'email' por tu email de Databricks")
else:
    # Configurar el experimento
    experiment_name = f"/Users/{email}/4-ejercicio-the-irish"
    mlflow.set_experiment(experiment_name)
    
    print("=" * 60)
    print("✅ MLflow configurado correctamente")
    print("=" * 60)
    print(f"📊 Experimento: {experiment_name}")
    print(f"🌸 Dataset: Iris (Clasificación)")
    print(f"🤖 Modelo: Decision Tree Classifier")
    print("=" * 60)

## ⚙️ Paso 1: Configuración de MLflow

Configuramos el experimento donde se registrarán todas las ejecuciones.

**🔴 IMPORTANTE**: Cambia el email vacío por tu email de Databricks.

## 📚 Paso 2: Importar Librerías

Importamos todas las herramientas necesarias para clasificación.

In [0]:
# Librerías principales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# MLflow para tracking
import mlflow
import mlflow.sklearn

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn import datasets
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    confusion_matrix,
    classification_report
)

# Configuración de visualización
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")

print("✅ Todas las librerías importadas correctamente")
print("   - MLflow: Listo para tracking")
print("   - Scikit-learn: Modelos y métricas")
print("   - Matplotlib & Seaborn: Visualizaciones")

## 🌺 Paso 3: Cargar y Explorar el Dataset Iris

Cargaremos el famoso dataset de flores Iris y exploraremos sus características.

In [0]:
# Cargar el dataset Iris
dataset = datasets.load_iris()

# Crear un DataFrame para mejor visualización
df = pd.DataFrame(
    data=dataset.data,
    columns=dataset.feature_names
)
df['species'] = dataset.target
df['species_name'] = df['species'].map({
    0: 'Setosa',
    1: 'Versicolor', 
    2: 'Virginica'
})

print("=" * 70)
print("🌸 DATASET IRIS - INFORMACIÓN GENERAL")
print("=" * 70)
print(f"📦 Número total de muestras: {len(df)}")
print(f"📊 Características (features): {len(dataset.feature_names)}")
print(f"🎯 Clases: {len(dataset.target_names)}")
print(f"\n🌺 Distribución de clases:")
print(df['species_name'].value_counts().to_string())

print(f"\n📏 Características del dataset:")
for i, feature in enumerate(dataset.feature_names, 1):
    print(f"   {i}. {feature}")

print("\n" + "=" * 70)
print("📊 PRIMERAS 5 MUESTRAS:")
print("=" * 70)
print(df.head().to_string())

print("\n" + "=" * 70)
print("📈 ESTADÍSTICAS DESCRIPTIVAS:")
print("=" * 70)
print(df.describe().to_string())

### 📊 Visualización Exploratoria del Dataset

Visualicemos las relaciones entre las características para entender mejor los datos.

In [ ]:
# Crear visualizaciones exploratorias
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('🌸 Análisis Exploratorio del Dataset Iris', fontsize=16, fontweight='bold')

# Gráfico 1: Pairplot de características más importantes
ax1 = plt.subplot(2, 2, 1)
for species_id, species_name in enumerate(dataset.target_names):
    mask = df['species'] == species_id
    ax1.scatter(
        df[mask]['petal length (cm)'],
        df[mask]['petal width (cm)'],
        label=species_name,
        alpha=0.7,
        s=100,
        edgecolors='black',
        linewidth=0.5
    )
ax1.set_xlabel('Petal Length (cm)', fontsize=11)
ax1.set_ylabel('Petal Width (cm)', fontsize=11)
ax1.set_title('Longitud vs Ancho del Pétalo', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Gráfico 2: Distribución de características por especie
ax2 = plt.subplot(2, 2, 2)
df_melted = df[['sepal length (cm)', 'sepal width (cm)', 
                'petal length (cm)', 'petal width (cm)', 'species_name']].melt(
    id_vars='species_name',
    var_name='feature',
    value_name='value'
)
sns.violinplot(data=df_melted, x='feature', y='value', hue='species_name', ax=ax2)
ax2.set_xlabel('Característica', fontsize=11)
ax2.set_ylabel('Valor (cm)', fontsize=11)
ax2.set_title('Distribución de Características por Especie', fontweight='bold')
ax2.tick_params(axis='x', rotation=45)
ax2.legend(title='Especie')

# Gráfico 3: Boxplot de longitud del pétalo
ax3 = plt.subplot(2, 2, 3)
df.boxplot(column='petal length (cm)', by='species_name', ax=ax3)
ax3.set_xlabel('Especie', fontsize=11)
ax3.set_ylabel('Petal Length (cm)', fontsize=11)
ax3.set_title('Distribución de Longitud del Pétalo', fontweight='bold')
plt.sca(ax3)
plt.xticks(rotation=45)

# Gráfico 4: Correlación entre características
ax4 = plt.subplot(2, 2, 4)
correlation = df[dataset.feature_names].corr()
sns.heatmap(correlation, annot=True, fmt='.2f', cmap='coolwarm', ax=ax4,
            cbar_kws={'label': 'Correlación'})
ax4.set_title('Matriz de Correlación', fontweight='bold')

plt.tight_layout()
plt.show()

print("✅ Visualizaciones generadas")
print("\n🔍 Observaciones clave:")
print("   - Las características del pétalo separan mejor las especies")
print("   - Setosa es claramente separable de las otras dos")
print("   - Versicolor y Virginica tienen mayor solapamiento")

## 🔀 Paso 4: Preparar los Datos

Dividimos el dataset en conjuntos de entrenamiento y prueba.

In [0]:
# Separar características (X) y etiquetas (y)
X = dataset.data
y = dataset.target

# Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,      # 25% para prueba
    random_state=42,     # Reproducibilidad
    stratify=y           # Mantener proporción de clases
)

print("=" * 60)
print("🔀 DIVISIÓN DEL DATASET")
print("=" * 60)
print(f"📊 Total de muestras: {len(X)}")
print(f"\n📚 Conjunto de Entrenamiento:")
print(f"   - Muestras: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"   - Shape: {X_train.shape}")
print(f"   - Distribución de clases:")
unique, counts = np.unique(y_train, return_counts=True)
for class_id, count in zip(unique, counts):
    print(f"     • {dataset.target_names[class_id]}: {count} muestras")

print(f"\n🧪 Conjunto de Prueba:")
print(f"   - Muestras: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")
print(f"   - Shape: {X_test.shape}")
print(f"   - Distribución de clases:")
unique, counts = np.unique(y_test, return_counts=True)
for class_id, count in zip(unique, counts):
    print(f"     • {dataset.target_names[class_id]}: {count} muestras")

print("=" * 60)
print("✅ Datos preparados para entrenamiento")

## 🌳 Paso 5: Entrenar Árbol de Decisión con MLflow

Ahora entrenaremos un modelo de **Decision Tree** y registraremos todo con MLflow.

### 🔑 Hiperparámetros del Árbol de Decisión

- **max_depth**: Profundidad máxima del árbol (evita overfitting)
- **max_features**: Número máximo de características a considerar por split
- Valores más altos = modelo más complejo = mayor riesgo de overfitting

In [0]:
# ========================================
# ENTRENAMIENTO CON MLFLOW
# ========================================

# Activar autologging de MLflow
mlflow.sklearn.autolog()

# Iniciar run con nombre descriptivo
with mlflow.start_run(run_name="Decision Tree - Iris Classifier") as run:
    
    print("=" * 70)
    print("🚀 ENTRENAMIENTO DEL ÁRBOL DE DECISIÓN")
    print("=" * 70)
    
    # =====================================
    # 1. DEFINIR HIPERPARÁMETROS
    # =====================================
    max_depth = 10        # Profundidad máxima del árbol
    max_features = 2      # Número máximo de características por split
    min_samples_split = 2 # Mínimo de muestras para hacer un split
    min_samples_leaf = 1  # Mínimo de muestras en hoja
    random_state = 42     # Reproducibilidad
    
    print("\n⚙️  HIPERPARÁMETROS:")
    print(f"   - max_depth: {max_depth}")
    print(f"   - max_features: {max_features}")
    print(f"   - min_samples_split: {min_samples_split}")
    print(f"   - min_samples_leaf: {min_samples_leaf}")
    
    # =====================================
    # 2. CREAR Y ENTRENAR EL MODELO
    # =====================================
    print("\n🌳 Creando Árbol de Decisión...")
    dt = DecisionTreeClassifier(
        max_depth=max_depth,
        max_features=max_features,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=random_state
    )
    
    print("📚 Entrenando con", len(X_train), "muestras...")
    dt.fit(X_train, y_train)
    print("✅ Modelo entrenado exitosamente")
    
    # =====================================
    # 3. HACER PREDICCIONES
    # =====================================
    print("\n🔮 Realizando predicciones...")
    y_pred_train = dt.predict(X_train)
    y_pred_test = dt.predict(X_test)
    
    # Probabilidades para análisis adicional
    y_pred_proba_test = dt.predict_proba(X_test)
    
    print("✅ Predicciones completadas")
    
    # =====================================
    # 4. CALCULAR MÉTRICAS
    # =====================================
    print("\n📊 CALCULANDO MÉTRICAS DE CLASIFICACIÓN:")
    
    # Métricas de entrenamiento
    accuracy_train = accuracy_score(y_train, y_pred_train)
    precision_train = precision_score(y_train, y_pred_train, average='weighted')
    recall_train = recall_score(y_train, y_pred_train, average='weighted')
    f1_train = f1_score(y_train, y_pred_train, average='weighted')
    
    # Métricas de prueba
    accuracy_test = accuracy_score(y_test, y_pred_test)
    precision_test = precision_score(y_test, y_pred_test, average='weighted')
    recall_test = recall_score(y_test, y_pred_test, average='weighted')
    f1_test = f1_score(y_test, y_pred_test, average='weighted')
    
    # Cross-validation
    cv_scores = cross_val_score(dt, X_train, y_train, cv=5)
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()
    
    print("\n   📚 ENTRENAMIENTO:")
    print(f"      - Accuracy:  {accuracy_train:.4f} ({accuracy_train*100:.2f}%)")
    print(f"      - Precision: {precision_train:.4f}")
    print(f"      - Recall:    {recall_train:.4f}")
    print(f"      - F1-Score:  {f1_train:.4f}")
    
    print("\n   🧪 PRUEBA:")
    print(f"      - Accuracy:  {accuracy_test:.4f} ({accuracy_test*100:.2f}%)")
    print(f"      - Precision: {precision_test:.4f}")
    print(f"      - Recall:    {recall_test:.4f}")
    print(f"      - F1-Score:  {f1_test:.4f}")
    
    print(f"\n   🔄 CROSS-VALIDATION (5-fold):")
    print(f"      - Mean Accuracy: {cv_mean:.4f} ± {cv_std:.4f}")
    
    # Calcular overfitting
    overfitting = accuracy_train - accuracy_test
    print(f"\n   ⚠️  Overfitting Score: {overfitting:.4f}")
    if overfitting < 0.05:
        print("      ✅ Excelente - Poco overfitting")
    elif overfitting < 0.10:
        print("      ⚡ Bueno - Overfitting moderado")
    else:
        print("      ⚠️  Cuidado - Posible overfitting")
    
    # =====================================
    # 5. REGISTRAR EN MLFLOW
    # =====================================
    print("\n📝 Registrando información en MLflow...")
    
    # Registrar hiperparámetros adicionales
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("max_features", max_features)
    mlflow.log_param("min_samples_split", min_samples_split)
    mlflow.log_param("min_samples_leaf", min_samples_leaf)
    mlflow.log_param("dataset", "Iris")
    mlflow.log_param("n_classes", len(dataset.target_names))
    
    # Registrar métricas
    mlflow.log_metric("train_accuracy", accuracy_train)
    mlflow.log_metric("test_accuracy", accuracy_test)
    mlflow.log_metric("train_precision", precision_train)
    mlflow.log_metric("test_precision", precision_test)
    mlflow.log_metric("train_recall", recall_train)
    mlflow.log_metric("test_recall", recall_test)
    mlflow.log_metric("train_f1", f1_train)
    mlflow.log_metric("test_f1", f1_test)
    mlflow.log_metric("cv_mean_accuracy", cv_mean)
    mlflow.log_metric("cv_std_accuracy", cv_std)
    mlflow.log_metric("overfitting_score", overfitting)
    
    # =====================================
    # 6. CREAR VISUALIZACIONES
    # =====================================
    print("\n📈 Generando visualizaciones...")
    
    # Figura con múltiples análisis
    fig = plt.figure(figsize=(16, 12))
    
    # 1. Matriz de Confusión
    ax1 = plt.subplot(2, 3, 1)
    cm = confusion_matrix(y_test, y_pred_test)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
                xticklabels=dataset.target_names,
                yticklabels=dataset.target_names)
    ax1.set_title('Matriz de Confusión', fontweight='bold', fontsize=12)
    ax1.set_ylabel('Verdadero', fontsize=10)
    ax1.set_xlabel('Predicción', fontsize=10)
    
    # 2. Visualización del Árbol
    ax2 = plt.subplot(2, 3, 2)
    plot_tree(dt, 
              feature_names=dataset.feature_names,
              class_names=dataset.target_names,
              filled=True,
              rounded=True,
              ax=ax2,
              fontsize=8)
    ax2.set_title('Estructura del Árbol de Decisión', fontweight='bold', fontsize=12)
    
    # 3. Importancia de Características
    ax3 = plt.subplot(2, 3, 3)
    feature_importance = pd.DataFrame({
        'feature': dataset.feature_names,
        'importance': dt.feature_importances_
    }).sort_values('importance', ascending=True)
    ax3.barh(feature_importance['feature'], feature_importance['importance'])
    ax3.set_xlabel('Importancia', fontsize=10)
    ax3.set_title('Importancia de Características', fontweight='bold', fontsize=12)
    ax3.grid(True, alpha=0.3, axis='x')
    
    # 4. Comparación de Métricas
    ax4 = plt.subplot(2, 3, 4)
    metrics_comparison = pd.DataFrame({
        'Métrica': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
        'Entrenamiento': [accuracy_train, precision_train, recall_train, f1_train],
        'Prueba': [accuracy_test, precision_test, recall_test, f1_test]
    })
    x = np.arange(len(metrics_comparison))
    width = 0.35
    ax4.bar(x - width/2, metrics_comparison['Entrenamiento'], width, label='Entrenamiento', alpha=0.8)
    ax4.bar(x + width/2, metrics_comparison['Prueba'], width, label='Prueba', alpha=0.8)
    ax4.set_xlabel('Métrica', fontsize=10)
    ax4.set_ylabel('Valor', fontsize=10)
    ax4.set_title('Comparación Train vs Test', fontweight='bold', fontsize=12)
    ax4.set_xticks(x)
    ax4.set_xticklabels(metrics_comparison['Métrica'], rotation=45)
    ax4.legend()
    ax4.grid(True, alpha=0.3, axis='y')
    ax4.set_ylim([0, 1.1])
    
    # 5. Predicciones por clase
    ax5 = plt.subplot(2, 3, 5)
    class_accuracy = []
    for i, class_name in enumerate(dataset.target_names):
        mask = y_test == i
        if mask.sum() > 0:
            acc = accuracy_score(y_test[mask], y_pred_test[mask])
            class_accuracy.append(acc)
        else:
            class_accuracy.append(0)
    
    bars = ax5.bar(dataset.target_names, class_accuracy, alpha=0.7, edgecolor='black')
    for bar, acc in zip(bars, class_accuracy):
        height = bar.get_height()
        ax5.text(bar.get_x() + bar.get_width()/2., height,
                f'{acc:.2%}', ha='center', va='bottom', fontsize=10)
    ax5.set_ylabel('Accuracy', fontsize=10)
    ax5.set_title('Accuracy por Clase', fontweight='bold', fontsize=12)
    ax5.set_ylim([0, 1.1])
    ax5.grid(True, alpha=0.3, axis='y')
    
    # 6. Distribución de confianza en predicciones
    ax6 = plt.subplot(2, 3, 6)
    max_probas = np.max(y_pred_proba_test, axis=1)
    ax6.hist(max_probas, bins=20, edgecolor='black', alpha=0.7)
    ax6.axvline(max_probas.mean(), color='red', linestyle='--', 
                linewidth=2, label=f'Media: {max_probas.mean():.3f}')
    ax6.set_xlabel('Confianza de Predicción', fontsize=10)
    ax6.set_ylabel('Frecuencia', fontsize=10)
    ax6.set_title('Distribución de Confianza', fontweight='bold', fontsize=12)
    ax6.legend()
    ax6.grid(True, alpha=0.3, axis='y')
    
    plt.suptitle('🌸 Análisis Completo del Modelo Decision Tree - Iris', 
                 fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    
    # Guardar en MLflow
    mlflow.log_figure(fig, "model_analysis_complete.png")
    plt.show()
    
    print("✅ Visualizaciones guardadas")
    
    # =====================================
    # 7. REPORTE DE CLASIFICACIÓN
    # =====================================
    print("\n" + "=" * 70)
    print("📋 REPORTE DE CLASIFICACIÓN DETALLADO")
    print("=" * 70)
    print(classification_report(
        y_test, 
        y_pred_test, 
        target_names=dataset.target_names,
        digits=4
    ))
    
    # =====================================
    # 8. INFORMACIÓN FINAL
    # =====================================
    run_id = run.info.run_id
    
    print("=" * 70)
    print("🎉 ENTRENAMIENTO COMPLETADO EXITOSAMENTE")
    print("=" * 70)
    print(f"🆔 Run ID: {run_id}")
    print(f"📊 Experimento: {experiment_name if email else 'No configurado'}")
    print(f"🎯 Accuracy (Test): {accuracy_test:.4f} ({accuracy_test*100:.2f}%)")
    print(f"🌟 F1-Score (Test): {f1_test:.4f}")
    print(f"📊 Cross-Val Accuracy: {cv_mean:.4f} ± {cv_std:.4f}")
    print("=" * 70)

mlflow.end_run()
print("\n✅ Experimento finalizado correctamente")

## 🎯 ¡Excelente Trabajo! Ejercicio Completado

### ✅ Lo que has Logrado

1. ✅ **Explorado el dataset Iris** con análisis visual completo
2. ✅ **Entrenado un Árbol de Decisión** para clasificación multiclase
3. ✅ **Evaluado con múltiples métricas** (Accuracy, Precision, Recall, F1)
4. ✅ **Implementado Cross-Validation** para validación robusta
5. ✅ **Registrado todo en MLflow** de forma profesional
6. ✅ **Creado visualizaciones avanzadas** (matriz de confusión, árbol, etc.)

---

## 📚 Conceptos Clave - Clasificación

### 🎯 Métricas de Clasificación

| Métrica | Descripción | Cuándo Usarla |
|---------|-------------|---------------|
| **Accuracy** | % de predicciones correctas | Clases balanceadas |
| **Precision** | % de positivos correctos | Importante evitar falsos positivos |
| **Recall** | % de positivos encontrados | Importante encontrar todos los positivos |
| **F1-Score** | Media armónica de P y R | Balance entre precisión y recall |

### 🌳 Árbol de Decisión

**Ventajas:**
- ✅ Fácil de interpretar y visualizar
- ✅ No requiere normalización de datos
- ✅ Maneja datos numéricos y categóricos
- ✅ Captura relaciones no lineales

**Desventajas:**
- ⚠️ Propenso a overfitting
- ⚠️ Inestable ante pequeños cambios
- ⚠️ Puede crear sesgos con clases desbalanceadas

### 📊 Matriz de Confusión

```
                Predicción
              Setosa  Versi  Virgin
Real Setosa     [TP]   [FP]   [FP]
     Versi      [FN]   [TP]   [FP]
     Virgin     [FN]   [FN]   [TP]
```

- **TP (True Positive)**: Predicción correcta
- **FP (False Positive)**: Error tipo I
- **FN (False Negative)**: Error tipo II

---

## 🚀 Desafíos Adicionales

### 🎯 Desafío 1: Optimiza los Hiperparámetros

Prueba diferentes configuraciones y encuentra la mejor:

```python
# Experimenta con:
max_depth = [3, 5, 10, None]
max_features = [2, 3, 4, 'sqrt']
min_samples_split = [2, 5, 10]
```

**Pregunta**: ¿Qué combinación da el mejor balance entre accuracy y overfitting?

---

### 🎯 Desafío 2: Implementa Grid Search

Automatiza la búsqueda del mejor modelo:

```python
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [3, 5, 10, None],
    'max_features': [2, 3, 4],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Completa el código...
```

---

### 🎯 Desafío 3: Compara con Otros Modelos

Entrena y compara:
- Random Forest
- SVM (Support Vector Machine)
- KNN (K-Nearest Neighbors)
- Logistic Regression

**Pregunta**: ¿Cuál funciona mejor para Iris? ¿Por qué?

---

### 🎯 Desafío 4: Análisis de Errores

Analiza las predicciones incorrectas:
- ¿Qué flores se confunden más?
- ¿Por qué características similares causan errores?
- ¿Cómo mejorarías el modelo?

---

### 🎯 Desafío 5: Reducción de Dimensionalidad

Usa solo 2 características y visualiza:

```python
from sklearn.decomposition import PCA

# Reducir a 2D
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# Entrenar y visualizar límites de decisión
```

---

## 💡 Tips para Mejorar

### 🎨 Mejora la Organización en MLflow

```python
# Agregar tags descriptivos
mlflow.set_tag("tipo_modelo", "decision_tree")
mlflow.set_tag("dataset", "iris")
mlflow.set_tag("objetivo", "clasificacion_multiclase")
mlflow.set_tag("version", "1.0")
mlflow.set_tag("autor", "tu_nombre")

# Agregar notas
mlflow.set_tag("notas", "Primer experimento baseline")
```

### 📊 Experimenta con Visualizaciones

```python
# Límites de decisión en 2D
from matplotlib.colors import ListedColormap
# ... código para visualizar límites ...

# Curvas de aprendizaje
from sklearn.model_selection import learning_curve
# ... código para curvas de aprendizaje ...
```

### 🔄 Validación Cruzada Estratificada

```python
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_scores = cross_val_score(dt, X, y, cv=skf)
```

---

## 🎓 Recursos Adicionales

### 📖 Documentación
- [Scikit-learn Decision Trees](https://scikit-learn.org/stable/modules/tree.html)
- [Classification Metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics)
- [MLflow Tracking](https://mlflow.org/docs/latest/tracking.html)

### 🎥 Tutoriales
- Interpretación de matrices de confusión
- Árboles de decisión explicados visualmente
- Optimización de hiperparámetros

### 📊 Datasets Similares
- **Wine Quality**: Clasificación de vinos
- **Digits**: Reconocimiento de dígitos escritos
- **Breast Cancer**: Diagnóstico médico

---

## 🌟 Próximos Pasos

1. **Completa los desafíos** propuestos
2. **Compara tus resultados** en MLflow UI
3. **Documenta tus hallazgos** con tags y notas
4. **Experimenta con otros datasets** de clasificación
5. **Aprende sobre ensembles** (Random Forest, Gradient Boosting)

---

## 🎉 ¡Felicitaciones!

Has completado con éxito el ejercicio de clasificación de Iris con MLflow. Ahora tienes las habilidades para:

- ✅ Trabajar con problemas de clasificación multiclase
- ✅ Evaluar modelos con métricas apropiadas
- ✅ Interpretar matrices de confusión
- ✅ Visualizar resultados de clasificación
- ✅ Documentar experimentos profesionalmente

**¡Sigue practicando y mejorando tus habilidades en Machine Learning! 🚀🌸**

In [ ]:
# 🎨 CÓDIGO DE EJEMPLO PARA LOS DESAFÍOS
# Descomenta y experimenta con diferentes configuraciones

# ========================================
# DESAFÍO 1: Optimización Manual
# ========================================
# configuraciones = [
#     {'max_depth': 3, 'max_features': 2},
#     {'max_depth': 5, 'max_features': 3},
#     {'max_depth': 10, 'max_features': 4},
#     {'max_depth': None, 'max_features': 'sqrt'},
# ]
# 
# mlflow.sklearn.autolog()
# for i, config in enumerate(configuraciones):
#     with mlflow.start_run(run_name=f"DT Config {i+1}"):
#         dt = DecisionTreeClassifier(**config, random_state=42)
#         dt.fit(X_train, y_train)
#         accuracy = accuracy_score(y_test, dt.predict(X_test))
#         print(f"Config {i+1}: {config} -> Accuracy: {accuracy:.4f}")

# ========================================
# DESAFÍO 2: Grid Search
# ========================================
# from sklearn.model_selection import GridSearchCV
# 
# param_grid = {
#     'max_depth': [3, 5, 10, None],
#     'max_features': [2, 3, 4],
#     'min_samples_split': [2, 5, 10],
#     'min_samples_leaf': [1, 2, 4]
# }
# 
# mlflow.sklearn.autolog()
# with mlflow.start_run(run_name="Grid Search - Decision Tree"):
#     grid_search = GridSearchCV(
#         DecisionTreeClassifier(random_state=42),
#         param_grid,
#         cv=5,
#         scoring='accuracy',
#         verbose=1,
#         n_jobs=-1
#     )
#     grid_search.fit(X_train, y_train)
#     
#     print(f"Mejores parámetros: {grid_search.best_params_}")
#     print(f"Mejor accuracy (CV): {grid_search.best_score_:.4f}")
#     print(f"Accuracy en test: {accuracy_score(y_test, grid_search.predict(X_test)):.4f}")

# ========================================
# DESAFÍO 3: Comparación de Modelos
# ========================================
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.svm import SVC
# from sklearn.neighbors import KNeighborsClassifier
# from sklearn.linear_model import LogisticRegression
# 
# modelos = {
#     'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
#     'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
#     'SVM': SVC(kernel='rbf', random_state=42),
#     'KNN': KNeighborsClassifier(n_neighbors=5),
#     'Logistic Regression': LogisticRegression(max_iter=200, random_state=42)
# }
# 
# resultados = []
# mlflow.sklearn.autolog()
# 
# for nombre, modelo in modelos.items():
#     with mlflow.start_run(run_name=f"{nombre} - Iris"):
#         modelo.fit(X_train, y_train)
#         y_pred = modelo.predict(X_test)
#         
#         acc = accuracy_score(y_test, y_pred)
#         f1 = f1_score(y_test, y_pred, average='weighted')
#         
#         resultados.append({
#             'Modelo': nombre,
#             'Accuracy': acc,
#             'F1-Score': f1
#         })
#         print(f"{nombre}: Accuracy={acc:.4f}, F1={f1:.4f}")
# 
# # Visualizar comparación
# df_resultados = pd.DataFrame(resultados).sort_values('Accuracy', ascending=False)
# print("\n📊 Ranking de Modelos:")
# print(df_resultados.to_string(index=False))

# ========================================
# DESAFÍO 4: Análisis de Errores
# ========================================
# # Encontrar predicciones incorrectas
# incorrect_indices = np.where(y_pred_test != y_test)[0]
# 
# print(f"Total de errores: {len(incorrect_indices)}")
# print("\nAnálisis de errores:")
# for idx in incorrect_indices:
#     true_class = dataset.target_names[y_test[idx]]
#     pred_class = dataset.target_names[y_pred_test[idx]]
#     print(f"  Muestra {idx}: Real={true_class}, Predicción={pred_class}")
#     print(f"    Características: {X_test[idx]}")

# ========================================
# DESAFÍO 5: PCA y Visualización 2D
# ========================================
# from sklearn.decomposition import PCA
# 
# # Reducir a 2 dimensiones
# pca = PCA(n_components=2)
# X_pca = pca.fit_transform(X)
# X_train_pca, X_test_pca, y_train_pca, y_test_pca = train_test_split(
#     X_pca, y, test_size=0.25, random_state=42, stratify=y
# )
# 
# # Entrenar modelo con datos reducidos
# dt_pca = DecisionTreeClassifier(max_depth=5, random_state=42)
# dt_pca.fit(X_train_pca, y_train_pca)
# 
# # Visualizar límites de decisión
# from matplotlib.colors import ListedColormap
# 
# h = 0.02  # step size
# x_min, x_max = X_pca[:, 0].min() - 1, X_pca[:, 0].max() + 1
# y_min, y_max = X_pca[:, 1].min() - 1, X_pca[:, 1].max() + 1
# xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
#                      np.arange(y_min, y_max, h))
# 
# Z = dt_pca.predict(np.c_[xx.ravel(), yy.ravel()])
# Z = Z.reshape(xx.shape)
# 
# plt.figure(figsize=(10, 8))
# plt.contourf(xx, yy, Z, alpha=0.4, cmap=plt.cm.RdYlBu)
# scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, 
#                       edgecolors='black', s=80, cmap=plt.cm.RdYlBu)
# plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} varianza)')
# plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} varianza)')
# plt.title('Límites de Decisión con PCA (2D)')
# plt.colorbar(scatter)
# plt.show()
# 
# print(f"Accuracy con PCA: {accuracy_score(y_test_pca, dt_pca.predict(X_test_pca)):.4f}")

print("💡 Descomenta y ejecuta el código del desafío que quieras explorar")